# Ejercicios Tema 3: Pandas

Estos ejercicios están separados de los de NumPy, que se encuentran en `EjerciciosTema3.ipynb`.

## Preparación de los datos

El primer ejercicio utiliza `titanic_demo.csv`. Guarda ese archivo en la misma carpeta que este notebook y la celda lo cargará automáticamente. Si todavía no tienes el CSV, se crea un pequeño DataFrame de demostración para poder ejecutar y entender los ejercicios. Los datos de demostración no sustituyen al dataset real.

Se espera que el CSV contenga las columnas `sex`, `age`, `pclass`, `fare`, `embarked` y `survived`.

In [1]:
from pathlib import Path
import pandas as pd

ruta_csv = Path("titanic_demo.csv")
columnas_necesarias = ["sex", "age", "pclass", "fare", "embarked", "survived"]

if ruta_csv.exists():
    df = pd.read_csv(ruta_csv)
    faltantes = [columna for columna in columnas_necesarias if columna not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas necesarias en {ruta_csv}: {faltantes}")
    df = df[columnas_necesarias].copy()
    print(f"Dataset cargado desde: {ruta_csv.resolve()}")
else:
    df = pd.DataFrame(
        {
            "sex": ["female", "male", "female", "male", "female", "male", "female", "male", "female", "male"],
            "age": [22.0, 17.0, None, 45.0, 8.0, 60.0, 19.0, 35.0, 28.0, 3.0],
            "pclass": [1, 3, 2, 1, 3, 1, 2, 3, 1, 2],
            "fare": [72.50, 7.25, 13.00, 80.00, 8.05, 55.00, 26.00, 9.50, 120.00, 15.00],
            "embarked": ["S", "S", "C", "C", "S", "Q", "S", "Q", "C", "S"],
            "survived": [1, 0, 1, 0, 1, 0, 1, 0, 1, 1],
        }
    )
    print("No se encontró titanic_demo.csv; se está usando un DataFrame pequeño de demostración.")

display(df.head())
print("Dimensiones del DataFrame:", df.shape)

No se encontró titanic_demo.csv; se está usando un DataFrame pequeño de demostración.


,sex,age,pclass,fare,embarked,survived
0,female,22.0,1,72.50,S,1
1,male,17.0,3,7.25,S,0
2,female,NaN,2,13.00,C,1
3,male,45.0,1,80.00,C,0
4,female,8.0,3,8.05,S,1


Dimensiones del DataFrame: (10, 6)


## Ejercicio 1: clasificación de pasajeros por edad

Crea la columna `age_label` con estas categorías:

- `"menor"` si la edad es menor que 18.
- `"adulto"` si la edad es igual o mayor que 18 y menor que 60.
- `"mayor"` si la edad es igual o mayor que 60.

Los valores nulos de `age` se etiquetan como `"desconocido"`; no se eliminan pasajeros ni se inventa una edad. Se inicializa la nueva columna con esa etiqueta y después `.loc` asigna las categorías a las filas que cumplen cada condición. Así, la edad 60 pertenece a `"mayor"`, tal como establece la regla `>= 60`.

In [ ]:
df["age_label"] = "desconocido"
df.loc[df["age"] < 18, "age_label"] = "menor"
df.loc[(df["age"] >= 18) & (df["age"] < 60), "age_label"] = "adulto"
df.loc[df["age"] >= 60, "age_label"] = "mayor"

display(df[["sex", "age", "age_label"]])

## Ejercicio 2: detección de tarifas extremas

Filtra por separado a los pasajeros con tarifa mayor que 50 y menor que 10. En ambos resultados muestra solo `sex`, `age` y `fare`. Las comparaciones no incluyen exactamente 50 ni exactamente 10, respectivamente.

In [ ]:
columnas_tarifa = ["sex", "age", "fare"]

tarifas_mayores_50 = df.loc[df["fare"] > 50, columnas_tarifa]
tarifas_menores_10 = df.loc[df["fare"] < 10, columnas_tarifa]

print("Pasajeros con tarifa mayor que 50:")
display(tarifas_mayores_50)

print("Pasajeros con tarifa menor que 10:")
display(tarifas_menores_10)

## Ejercicio 3: modificación condicional de tarifas

Crea `fare_adjusted`: para pasajeros de primera clase (`pclass == 1`), aumenta `fare` un 20 %; para el resto, conserva la tarifa original. La expresión `df["fare"] * 1.20` se aplica solo a las filas seleccionadas por `.loc`; las otras filas se copian desde `fare`.

In [ ]:
df["fare_adjusted"] = df["fare"]
primera_clase = df["pclass"] == 1
df.loc[primera_clase, "fare_adjusted"] = df.loc[primera_clase, "fare"] * 1.20

display(df[["pclass", "fare", "fare_adjusted"]])

## Ejercicio 4: selección combinada con `.loc[]`

Selecciona pasajeros con edad conocida y menor que 20 años. La condición `age.notna()` hace explícito que los nulos no cuentan como edades conocidas; además, las comparaciones de Pandas con valores nulos no los seleccionan. El resultado contiene solo `sex`, `age`, `pclass` y `embarked`.

In [ ]:
menores_de_20 = df.loc[
    df["age"].notna() & (df["age"] < 20),
    ["sex", "age", "pclass", "embarked"],
]

display(menores_de_20)

## Ejercicio 5: indexación mixta con `.iloc[]`

Selecciona las filas 3 a 7 y las columnas 2 a 5, contando posiciones desde 1 como en el enunciado. `.iloc[]` usa posiciones desde 0 y el extremo final de un slice no se incluye: por eso las filas se escriben `2:7` y las columnas `1:5`. El resultado contiene cinco filas y cuatro columnas, si el DataFrame tiene suficientes filas y columnas.

In [ ]:
filas_3_a_7_columnas_2_a_5 = df.iloc[2:7, 1:5]
display(filas_3_a_7_columnas_2_a_5)

# Concatenación de DataFrames

## Ejercicio 1: concatenación por columnas

A partir del dataset Titanic reducido, crea `df1` con `sex`, `age` y `pclass`, y `df2` con `fare`, `embarked` y `survived`. `pd.concat(..., axis=1)` los une horizontalmente. Pandas alinea las filas por su índice; por eso se conservan ambos DataFrames con el mismo índice.

In [ ]:
df1 = df[["sex", "age", "pclass"]].copy()
df2 = df[["fare", "embarked", "survived"]].copy()
titanic_combinado = pd.concat([df1, df2], axis=1)

display(titanic_combinado.head())
print("¿Se han conservado los índices?:", titanic_combinado.index.equals(df.index))

## Ejercicio 2: concatenación por filas y reinicio del índice

Concatena dos DataFrames que tienen las mismas columnas y observa que `concat` conserva las etiquetas de índice originales. Como ambos comienzan en el índice 0, al apilarlos aparecen índices repetidos. `reset_index(drop=True)` genera un índice nuevo consecutivo y descarta el índice anterior.

In [ ]:
pasajeros_a = pd.DataFrame(
    {"sex": ["female", "male"], "age": [29, 35], "pclass": [1, 2]}
)
pasajeros_b = pd.DataFrame(
    {"sex": ["male", "female"], "age": [18, 42], "pclass": [3, 1]}
)

pasajeros_apilados = pd.concat([pasajeros_a, pasajeros_b], axis=0)
print("Índices originales conservados (se repiten):")
display(pasajeros_apilados)

pasajeros_reindexados = pasajeros_apilados.reset_index(drop=True)
print("Índice nuevo, consecutivo:")
display(pasajeros_reindexados)

## Ejercicio 3 (extra): concatenación con columnas diferentes

Concatena por filas dos DataFrames que no comparten todas sus columnas. Pandas construye la unión de las columnas. Cuando una fila no tiene un valor para una de esas columnas, el resultado muestra un valor ausente (`NaN` en estas columnas numéricas).

In [ ]:
grupo_con_tarifa = pd.DataFrame(
    {"sex": ["female", "male"], "fare": [32.50, 8.00]}
)
grupo_con_embarque = pd.DataFrame(
    {"sex": ["female", "male"], "embarked": ["C", "S"]}
)

columnas_combinadas = pd.concat([grupo_con_tarifa, grupo_con_embarque], ignore_index=True)
display(columnas_combinadas)
print("Valores ausentes por columna:")
display(columnas_combinadas.isna().sum())